In [50]:
# %%
# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
from pathlib import Path
import json
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection")
sys.path.insert(0, str(ROOT))

from shapely.geometry import Point, Polygon
from scripts.shape_gen.intersections2 import find_intersection_points_multiple

In [51]:
# %%
# -----------------------------------------------------------------------------
# Paths and parameters
# -----------------------------------------------------------------------------
RESULTS_ROOT = Path(
    "/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/"
    "bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32"
)

GENERATED_CASES_ROOT = Path(
    "/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/data/cases/generated_from_best_occluders"
)

MANIFEST_CSV = RESULTS_ROOT / "mean_top1_cluster_contours_size32_manifest.csv"

N_POINTS_SHAPE = 1024
N_POINTS_OCCLUDER = 256
N_ARC = 250

SAVE_PLOT = True
SAVE_JSONL = True
SAVE_NPZ = True

print("RESULTS_ROOT exists:", RESULTS_ROOT.exists())
print("GENERATED_CASES_ROOT exists:", GENERATED_CASES_ROOT.exists())
print("MANIFEST_CSV:", MANIFEST_CSV)

RESULTS_ROOT exists: True
GENERATED_CASES_ROOT exists: True
MANIFEST_CSV: /home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32/mean_top1_cluster_contours_size32_manifest.csv


In [52]:
# %%
# -----------------------------------------------------------------------------
# Geometry helpers
# -----------------------------------------------------------------------------
def polygon_to_xy(poly, drop_duplicate_endpoint=True):
    xy = np.asarray(poly, dtype=float)
    if drop_duplicate_endpoint and len(xy) >= 2 and np.allclose(xy[0], xy[-1]):
        xy = xy[:-1]
    return xy


def signed_area(xy):
    xy = np.asarray(xy, dtype=float)
    if len(xy) < 3:
        return 0.0
    if not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])
    x = xy[:, 0]
    y = xy[:, 1]
    return 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])


def ensure_ccw(xy):
    xy = np.asarray(xy, dtype=float)
    if signed_area(xy) < 0:
        return xy[::-1].copy()
    return xy.copy()


def resample_contour_arc_length(xy, n_points=256, closed=True):
    xy = np.asarray(xy, dtype=float)

    if len(xy) < 2:
        raise ValueError("Contour must contain at least 2 points.")

    if closed and not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])

    seg = np.diff(xy, axis=0)
    seglen = np.sqrt((seg ** 2).sum(axis=1))

    keep = seglen > 1e-12
    if not np.all(keep):
        xy = np.vstack([xy[0], xy[1:][keep]])
        seg = np.diff(xy, axis=0)
        seglen = np.sqrt((seg ** 2).sum(axis=1))

    if len(seglen) == 0 or np.sum(seglen) <= 1e-12:
        raise ValueError("Contour has zero total arc length after removing duplicate points.")

    cum = np.concatenate([[0.0], np.cumsum(seglen)])
    total = float(cum[-1])

    if closed:
        t = np.linspace(0.0, total, n_points + 1)[:-1]
    else:
        t = np.linspace(0.0, total, n_points)

    x = np.interp(t, cum, xy[:, 0])
    y = np.interp(t, cum, xy[:, 1])
    return np.column_stack([x, y])

In [53]:
# %%
# -----------------------------------------------------------------------------
# Geometry helpers
# -----------------------------------------------------------------------------
def polygon_to_xy(poly, drop_duplicate_endpoint=True):
    xy = np.asarray(poly, dtype=float)
    if drop_duplicate_endpoint and len(xy) >= 2 and np.allclose(xy[0], xy[-1]):
        xy = xy[:-1]
    return xy


def signed_area(xy):
    xy = np.asarray(xy, dtype=float)
    if len(xy) < 3:
        return 0.0
    if not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])
    x = xy[:, 0]
    y = xy[:, 1]
    return 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])


def ensure_ccw(xy):
    xy = np.asarray(xy, dtype=float)
    if signed_area(xy) < 0:
        return xy[::-1].copy()
    return xy.copy()


def resample_contour_arc_length(xy, n_points=256, closed=True):
    xy = np.asarray(xy, dtype=float)

    if len(xy) < 2:
        raise ValueError("Contour must contain at least 2 points.")

    if closed and not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])

    seg = np.diff(xy, axis=0)
    seglen = np.sqrt((seg ** 2).sum(axis=1))

    keep = seglen > 1e-12
    if not np.all(keep):
        xy = np.vstack([xy[0], xy[1:][keep]])
        seg = np.diff(xy, axis=0)
        seglen = np.sqrt((seg ** 2).sum(axis=1))

    if len(seglen) == 0 or np.sum(seglen) <= 1e-12:
        raise ValueError("Contour has zero total arc length after removing duplicate points.")

    cum = np.concatenate([[0.0], np.cumsum(seglen)])
    total = float(cum[-1])

    if closed:
        t = np.linspace(0.0, total, n_points + 1)[:-1]
    else:
        t = np.linspace(0.0, total, n_points)

    x = np.interp(t, cum, xy[:, 0])
    y = np.interp(t, cum, xy[:, 1])
    return np.column_stack([x, y])

In [54]:
# %%
# -----------------------------------------------------------------------------
# Hidden-segment helpers
# -----------------------------------------------------------------------------
def point_to_segment_distance(p, a, b):
    p = np.asarray(p, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    ab = b - a
    denom = np.dot(ab, ab)
    if denom < 1e-12:
        return np.linalg.norm(p - a)

    t = np.dot(p - a, ab) / denom
    t = np.clip(t, 0.0, 1.0)
    proj = a + t * ab
    return np.linalg.norm(p - proj)


def insert_point_into_closed_polyline(xy, p, tol=1e-5):
    """
    Insert point p into the segment of closed polyline xy that contains it.
    xy is open-form closed contour, first point NOT repeated at end.
    """
    xy = np.asarray(xy, dtype=float)
    p = np.asarray(p, dtype=float)

    n = len(xy)
    best_i = None
    best_d = np.inf

    for i in range(n):
        a = xy[i]
        b = xy[(i + 1) % n]
        d = point_to_segment_distance(p, a, b)
        if d < best_d:
            best_d = d
            best_i = i

    if best_d > tol:
        raise ValueError(f"Point is not close enough to contour segment. best_d={best_d:.6g}")

    a = xy[best_i]
    b = xy[(best_i + 1) % n]

    if np.linalg.norm(a - p) < tol or np.linalg.norm(b - p) < tol:
        return xy.copy()

    return np.vstack([xy[:best_i + 1], p[None, :], xy[best_i + 1:]])


def index_of_point(xy, p, tol=1e-6):
    xy = np.asarray(xy, dtype=float)
    p = np.asarray(p, dtype=float)
    d = np.linalg.norm(xy - p[None, :], axis=1)
    i = np.argmin(d)
    if d[i] > tol:
        raise ValueError(f"Point not found in polyline. min_dist={d[i]:.6g}")
    return int(i)


def split_closed_contour_between_points(xy, p1, p2):
    """
    xy: open representation of closed contour, shape (N,2), first point not repeated
    returns the two candidate paths between p1 and p2
    """
    xy2 = insert_point_into_closed_polyline(xy, p1)
    xy2 = insert_point_into_closed_polyline(xy2, p2)

    i1 = index_of_point(xy2, p1)
    i2 = index_of_point(xy2, p2)

    if i1 > i2:
        i1, i2 = i2, i1
        p1, p2 = p2, p1

    path1 = xy2[i1:i2 + 1]
    path2 = np.vstack([xy2[i2:], xy2[:i1 + 1]])
    return path1, path2


def inside_fraction(path, poly):
    flags = np.array([poly.covers(Point(x, y)) for x, y in path], dtype=bool)
    return float(flags.mean())


def orient_arc_like_reference(arc, ref_start, ref_end):
    arc = np.asarray(arc, dtype=float)
    ref_start = np.asarray(ref_start, dtype=float)
    ref_end = np.asarray(ref_end, dtype=float)

    d_forward = np.sum((arc[0] - ref_start) ** 2) + np.sum((arc[-1] - ref_end) ** 2)
    d_reverse = np.sum((arc[-1] - ref_start) ** 2) + np.sum((arc[0] - ref_end) ** 2)

    if d_reverse < d_forward:
        return arc[::-1].copy()
    return arc.copy()


def orient_open_arc_to_endpoints(arc, start_xy, end_xy):
    return orient_arc_like_reference(arc, start_xy, end_xy)


def concat_open_arcs(a, b, tol=1e-8):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if len(a) == 0:
        return b.copy()
    if len(b) == 0:
        return a.copy()

    if np.linalg.norm(a[-1] - b[0]) <= tol:
        return np.vstack([a, b[1:]])
    return np.vstack([a, b])


def close_polyline(xy, tol=1e-8):
    xy = np.asarray(xy, dtype=float)
    if len(xy) == 0:
        return xy.copy()
    if np.linalg.norm(xy[0] - xy[-1]) <= tol:
        return xy.copy()
    return np.vstack([xy, xy[0]])


def extract_hidden_segment_from_intersections(contour_xy, occ_xy, occ_poly):
    """
    Extract the contour segment between the two true contour-occluder intersections.
    Chooses the candidate path that lies mostly inside the occluder.
    """
    contour_xy = np.asarray(contour_xy, dtype=float)
    occ_xy = np.asarray(occ_xy, dtype=float)

    inter_pts = find_intersection_points_multiple(contour_xy, occ_xy)

    if inter_pts.shape[0] != 2:
        return None, None, None

    p1, p2 = inter_pts[0], inter_pts[1]

    path_a, path_b = split_closed_contour_between_points(contour_xy, p1, p2)

    frac_a = inside_fraction(path_a, occ_poly)
    frac_b = inside_fraction(path_b, occ_poly)

    hidden = path_a if frac_a >= frac_b else path_b
    return hidden, p1, p2

In [55]:
# %%
# -----------------------------------------------------------------------------
# File readers
# -----------------------------------------------------------------------------
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def first_jsonl_row(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                return json.loads(line)
    raise ValueError(f"No rows found in {path}")

In [56]:
# %%
# -----------------------------------------------------------------------------
# Config / path resolution helpers
# -----------------------------------------------------------------------------
def resolve_case_jsonl_path_from_config(cfg):
    """
    Config may store an absolute path like /data/storage-occ-v2/...
    On this machine the accessible prefix is /home/hschatzle/data/storage-occ-v2/...
    """
    raw = Path(cfg["case_jsonl"])
    if raw.exists():
        return raw

    raw_str = str(raw)
    if raw_str.startswith("/data/storage-occ-v2/"):
        mapped = Path("/home/hschatzle") / raw_str.lstrip("/")
        if mapped.exists():
            return mapped

    raise FileNotFoundError(f"Could not resolve case_jsonl path from config: {raw}")


def load_generated_case_assets(case_id, model_type="resnet50_geirhos_tl", occluder_size=32):
    generated_dir = (
        GENERATED_CASES_ROOT
        / case_id
        / model_type
        / f"size_{int(occluder_size)}"
        / "generated"
    )

    shapes_npz_path = generated_dir / "shapes_xy.npz"
    source_occluder_config_path = generated_dir / "source_occluder_config.json"

    if not shapes_npz_path.exists():
        raise FileNotFoundError(f"Missing shapes_xy.npz: {shapes_npz_path}")
    if not source_occluder_config_path.exists():
        raise FileNotFoundError(f"Missing source_occluder_config.json: {source_occluder_config_path}")

    with open(source_occluder_config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    case_jsonl_path = resolve_case_jsonl_path_from_config(cfg)
    if not case_jsonl_path.exists():
        raise FileNotFoundError(f"Resolved case JSONL does not exist: {case_jsonl_path}")

    return {
        "generated_dir": generated_dir,
        "shapes_npz_path": shapes_npz_path,
        "source_occluder_config_path": source_occluder_config_path,
        "case_jsonl_path": case_jsonl_path,
        "cfg": cfg,
    }

In [57]:
# %%
# -----------------------------------------------------------------------------
# Core per-case function
# -----------------------------------------------------------------------------
def process_case(case_dir: Path):
    case_id = case_dir.name
    top10_jsonl_path = case_dir / "top10_clusters.jsonl"

    if not top10_jsonl_path.exists():
        raise FileNotFoundError(f"Missing top10_clusters.jsonl: {top10_jsonl_path}")

    first_cluster = first_jsonl_row(top10_jsonl_path)

    if int(first_cluster["cluster_rank"]) != 1:
        raise ValueError(f"First row is not cluster_rank=1 in {top10_jsonl_path}")

    target_k = int(first_cluster["k"])
    shapes_meta = first_cluster.get("shapes", [])
    if len(shapes_meta) == 0:
        raise ValueError(f"No shapes listed in top1 cluster for case {case_id}")

    assets = load_generated_case_assets(
        case_id=case_id,
        model_type="resnet50_geirhos_tl",
        occluder_size=32,
    )
    cfg = assets["cfg"]
    case_jsonl_path = assets["case_jsonl_path"]
    shapes_npz_path = assets["shapes_npz_path"]
    source_occluder_config_path = assets["source_occluder_config_path"]

    # Load original case geometry
    case_rows = read_jsonl(case_jsonl_path)
    if len(case_rows) == 0:
        raise ValueError(f"Empty case jsonl: {case_jsonl_path}")
    last_case_row = case_rows[-1]

    # Load generated shapes
    npz_meta = np.load(shapes_npz_path, allow_pickle=True)
    polygons = npz_meta["polygons"]
    baseGrid_case = float(np.asarray(npz_meta["base_grid"]).squeeze())

    # Base silhouette from original case jsonl
    sil_u = np.asarray(last_case_row["shape_contour_xy"], dtype=float)

    # Occluder from generated config, using normalized corners
    x0_norm = float(cfg["x0_norm"])
    y0_norm = float(cfg["y0_norm"])
    x1_norm = float(cfg["x1_norm"])
    y1_norm = float(cfg["y1_norm"])

    occ_u = np.array([
        [x0_norm, y0_norm],
        [x1_norm, y0_norm],
        [x1_norm, y1_norm],
        [x0_norm, y1_norm],
    ], dtype=float)

    sil_px = ensure_ccw(sil_u * baseGrid_case)
    occ_px = ensure_ccw(occ_u * baseGrid_case)

    occ_px_resampled = resample_contour_arc_length(
        occ_px,
        n_points=N_POINTS_OCCLUDER,
        closed=True,
    )

    occ_poly = Polygon(occ_px_resampled)
    if not occ_poly.is_valid:
        occ_poly = occ_poly.buffer(0)

    occ_closed = np.vstack([occ_px_resampled, occ_px_resampled[0]])

    # Top1 cluster shapes: use exact listed members
    global_indices = [int(x["global_index"]) for x in shapes_meta]
    eligible_rows = [int(x["eligible_row"]) for x in shapes_meta]
    resps = [float(x["resp"]) for x in shapes_meta]

    contours_raw = []
    for idx in global_indices:
        xy = polygon_to_xy(polygons[idx], drop_duplicate_endpoint=True)
        xy = ensure_ccw(xy)
        contours_raw.append(xy)

    # Reference GT hidden arc from original silhouette and generated occluder
    sil_res = resample_contour_arc_length(sil_px, n_points=N_POINTS_SHAPE, closed=True)

    gt_hidden_arc, gt_p1, gt_p2 = extract_hidden_segment_from_intersections(
        sil_res,
        occ_px_resampled,
        occ_poly,
    )

    if gt_hidden_arc is None:
        raise ValueError(f"Could not extract GT hidden arc for case {case_id}")

    common_start = gt_hidden_arc[0]
    common_end = gt_hidden_arc[-1]

    # Extract hidden arcs from cluster-member completion contours
    hidden_arcs = []
    arc_records = []

    for gidx, erow, resp, c in zip(global_indices, eligible_rows, resps, contours_raw):
        try:
            c_res = resample_contour_arc_length(c, n_points=N_POINTS_SHAPE, closed=True)
            arc, p1, p2 = extract_hidden_segment_from_intersections(
                c_res,
                occ_px_resampled,
                occ_poly,
            )

            if arc is None or len(arc) < 2:
                arc_records.append({
                    "case_id": case_id,
                    "target_k": target_k,
                    "global_index": int(gidx),
                    "eligible_row": int(erow),
                    "resp": float(resp),
                    "arc_extracted": False,
                    "inside_len": 0,
                })
                continue

            arc = orient_arc_like_reference(arc, common_start, common_end)
            hidden_arcs.append(arc)

            arc_records.append({
                "case_id": case_id,
                "target_k": target_k,
                "global_index": int(gidx),
                "eligible_row": int(erow),
                "resp": float(resp),
                "arc_extracted": True,
                "inside_len": int(len(arc)),
            })

        except Exception:
            arc_records.append({
                "case_id": case_id,
                "target_k": target_k,
                "global_index": int(gidx),
                "eligible_row": int(erow),
                "resp": float(resp),
                "arc_extracted": False,
                "inside_len": 0,
            })

    arc_df = pd.DataFrame(arc_records)

    if len(hidden_arcs) == 0:
        raise ValueError(f"No hidden arcs extracted for case {case_id}")

    # Resample arcs to common length and average
    hidden_arcs_resampled = []
    for arc in hidden_arcs:
        arc_res = resample_contour_arc_length(arc, n_points=N_ARC, closed=False)
        arc_res[0] = common_start
        arc_res[-1] = common_end
        hidden_arcs_resampled.append(arc_res)

    hidden_arcs_resampled = np.stack(hidden_arcs_resampled, axis=0)
    mean_hidden_arc = hidden_arcs_resampled.mean(axis=0)
    mean_hidden_arc[0] = common_start
    mean_hidden_arc[-1] = common_end

    # Recompute GT visible remainder to create full mean contour
    path_a, path_b = split_closed_contour_between_points(sil_res, gt_p1, gt_p2)
    frac_a = inside_fraction(path_a, occ_poly)
    frac_b = inside_fraction(path_b, occ_poly)

    if frac_a >= frac_b:
        gt_hidden_raw = path_a
        gt_visible_raw = path_b
    else:
        gt_hidden_raw = path_b
        gt_visible_raw = path_a

    gt_hidden_arc_save = orient_open_arc_to_endpoints(gt_hidden_raw, common_start, common_end)
    gt_visible_arc_save = orient_open_arc_to_endpoints(gt_visible_raw, common_end, common_start)
    mean_hidden_arc_save = orient_open_arc_to_endpoints(mean_hidden_arc, common_start, common_end)

    global_shape_filled_mean = concat_open_arcs(mean_hidden_arc_save, gt_visible_arc_save)
    global_shape_filled_mean = close_polyline(global_shape_filled_mean)

    intersection_points_xy = np.vstack([common_start, common_end])

    # Output names
    base_name = f"{case_id}_size32_top1cluster_mean_contour_k{target_k}"

    jsonl_path = case_dir / f"{base_name}.jsonl"
    npz_path = case_dir / f"{base_name}.npz"
    plot_path = case_dir / f"{base_name}.png"
    members_csv_path = case_dir / f"{base_name}_members.csv"

    # Save member extraction details
    arc_df.to_csv(members_csv_path, index=False)

    record = {
        "case_id": case_id,
        "source_model": "resnet50_geirhos_tl",
        "occluder_size": 32,
        "cluster_rank": 1,
        "target_k": int(target_k),
        "n_shapes_listed_in_cluster": int(len(shapes_meta)),
        "n_shapes_averaged": int(len(hidden_arcs_resampled)),
        "baseGrid_case": float(baseGrid_case),

        "source_occluder_config_path": str(source_occluder_config_path),
        "original_case_jsonl_path": str(case_jsonl_path),

        "gt_hidden_segment_xy": gt_hidden_arc_save.tolist(),
        "occluder_xy": occ_px_resampled.tolist(),
        "intersection_points_xy": intersection_points_xy.tolist(),
        "mean_hidden_segment_xy": mean_hidden_arc_save.tolist(),
        "global_shape_filled_mean_xy": global_shape_filled_mean.tolist(),
    }

    if SAVE_JSONL:
        with open(jsonl_path, "w", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

    if SAVE_NPZ:
        np.savez_compressed(
            npz_path,
            case_id=np.array(case_id),
            source_model=np.array("resnet50_geirhos_tl"),
            occluder_size=np.array(32),
            cluster_rank=np.array(1),
            target_k=np.array(target_k),
            baseGrid_case=np.array(baseGrid_case),
            gt_hidden_segment_xy=gt_hidden_arc_save,
            occluder_xy=occ_px_resampled,
            intersection_points_xy=intersection_points_xy,
            mean_hidden_segment_xy=mean_hidden_arc_save,
            global_shape_filled_mean_xy=global_shape_filled_mean,
            hidden_arcs_resampled=hidden_arcs_resampled,
        )

    if SAVE_PLOT:
        gt_arc_plot = resample_contour_arc_length(gt_hidden_arc, n_points=N_ARC, closed=False)

        fig, ax = plt.subplots(figsize=(8, 8))

        ax.plot(
            sil_px[:, 0],
            sil_px[:, 1],
            color="0.85",
            lw=1.2,
            label="reference silhouette",
        )

        ax.plot(
            occ_closed[:, 0],
            occ_closed[:, 1],
            "--",
            color="0.5",
            lw=1.2,
            label="occluder",
        )

        for arc in hidden_arcs_resampled:
            ax.plot(
                arc[:, 0],
                arc[:, 1],
                alpha=0.10,
                lw=1.0,
                color="tab:orange",
            )

        ax.plot(
            gt_arc_plot[:, 0],
            gt_arc_plot[:, 1],
            color="black",
            lw=3,
            label="GT hidden arc",
        )

        ax.plot(
            mean_hidden_arc[:, 0],
            mean_hidden_arc[:, 1],
            color="blue",
            lw=3,
            label="mean top1-cluster arc",
        )

        ax.scatter(
            [common_start[0], common_end[0]],
            [common_start[1], common_end[1]],
            color="red",
            s=45,
            zorder=5,
            label="anchors",
        )

        ax.set_aspect("equal")
        ax.set_title(f"{case_id} | size 32 | top1 cluster mean contour")
        ax.legend()
        fig.tight_layout()
        fig.savefig(plot_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

    return {
        "case_id": case_id,
        "source_model": "resnet50_geirhos_tl",
        "occluder_size": 16,
        "cluster_rank": 1,
        "target_k": int(target_k),
        "n_shapes_listed_in_cluster": int(len(shapes_meta)),
        "n_shapes_averaged": int(len(hidden_arcs_resampled)),
        "n_shapes_failed": int(len(shapes_meta) - len(hidden_arcs_resampled)),
        "baseGrid_case": float(baseGrid_case),
        "jsonl_path": str(jsonl_path) if SAVE_JSONL else "",
        "npz_path": str(npz_path) if SAVE_NPZ else "",
        "plot_path": str(plot_path) if SAVE_PLOT else "",
        "members_csv_path": str(members_csv_path),
        "status": "ok",
        "error": "",
    }

In [58]:
# %%
# -----------------------------------------------------------------------------
# One-case test first
# -----------------------------------------------------------------------------
TEST_CASE_ID = "ns_cow_202"
test_case_dir = RESULTS_ROOT / TEST_CASE_ID

test_row = process_case(test_case_dir)
pd.DataFrame([test_row])

,case_id,source_model,occluder_size,cluster_rank,target_k,n_shapes_listed_in_cluster,n_shapes_averaged,n_shapes_failed,baseGrid_case,jsonl_path,npz_path,plot_path,members_csv_path,status,error
0,ns_cow_202,resnet50_geirhos_tl,16,1,72,25,25,0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,


In [59]:
# %%
# Optional sanity check
test_base = f"{TEST_CASE_ID}_size32_top1cluster_mean_contour_k{test_row['target_k']}"
test_members_csv = test_case_dir / f"{test_base}_members.csv"

test_members_df = pd.read_csv(test_members_csv)
display(test_members_df.head())
print("Extracted arcs:", int(test_members_df["arc_extracted"].sum()), "/", len(test_members_df))

,case_id,target_k,global_index,eligible_row,resp,arc_extracted,inside_len
0,ns_cow_202,72,6769,6769,1.0,True,35
1,ns_cow_202,72,8494,8494,1.0,True,34
2,ns_cow_202,72,5292,5292,1.0,True,37
3,ns_cow_202,72,4032,4032,1.0,True,37
4,ns_cow_202,72,4234,4234,1.0,True,35


Extracted arcs: 25 / 25


In [60]:
# %%
# -----------------------------------------------------------------------------
# Batch run over all case folders
# -----------------------------------------------------------------------------
case_dirs = sorted([p for p in RESULTS_ROOT.iterdir() if p.is_dir()])

print(f"Found {len(case_dirs)} case folders under:\n{RESULTS_ROOT}")

manifest_rows = []

for i, case_dir in enumerate(case_dirs, start=1):
    case_id = case_dir.name
    print(f"[{i:03d}/{len(case_dirs):03d}] Processing {case_id}")

    try:
        row = process_case(case_dir)
        manifest_rows.append(row)
        print(
            f"  ok | k={row['target_k']} | "
            f"averaged={row['n_shapes_averaged']}/{row['n_shapes_listed_in_cluster']}"
        )
    except Exception as e:
        manifest_rows.append({
            "case_id": case_id,
            "source_model": "resnet50_geirhos_tl",
            "occluder_size": 16,
            "cluster_rank": 1,
            "target_k": np.nan,
            "n_shapes_listed_in_cluster": np.nan,
            "n_shapes_averaged": 0,
            "n_shapes_failed": np.nan,
            "baseGrid_case": np.nan,
            "jsonl_path": "",
            "npz_path": "",
            "plot_path": "",
            "members_csv_path": "",
            "status": "failed",
            "error": str(e),
        })
        print(f"  failed | {e}")
        traceback.print_exc()

Found 40 case folders under:
/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32
[001/040] Processing ns_cow_202
  ok | k=72 | averaged=25/25
[002/040] Processing ns_cow_205
  ok | k=70 | averaged=25/25
[003/040] Processing ns_cow_350
  ok | k=56 | averaged=25/25
[004/040] Processing ns_cow_510
  ok | k=36 | averaged=25/25
[005/040] Processing ns_cow_780
  ok | k=51 | averaged=25/25
[006/040] Processing ns_dog_105
  ok | k=76 | averaged=25/25
[007/040] Processing ns_dog_214
  ok | k=9 | averaged=25/25
[008/040] Processing ns_dog_350
  ok | k=41 | averaged=25/25
[009/040] Processing ns_dog_69
  ok | k=4 | averaged=25/25
[010/040] Processing ns_dog_780
  failed | Missing shapes_xy.npz: /home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/data/cases/generated_from_best_occluders/ns_dog_780/resnet50_geirhos_tl/size_32/generated/shapes_xy.npz
[011/040] Processing ns_elephant_100


Traceback (most recent call last):
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2645192461.py", line 16, in <module>
    row = process_case(case_dir)
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2920481930.py", line 22, in process_case
    assets = load_generated_case_assets(
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/3293915778.py", line 36, in load_generated_case_assets
    raise FileNotFoundError(f"Missing shapes_xy.npz: {shapes_npz_path}")
FileNotFoundError: Missing shapes_xy.npz: /home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/data/cases/generated_from_best_occluders/ns_dog_780/resnet50_geirhos_tl/size_32/generated/shapes_xy.npz


  failed | No hidden arcs extracted for case ns_elephant_100
[012/040] Processing ns_elephant_109


Traceback (most recent call last):
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2645192461.py", line 16, in <module>
    row = process_case(case_dir)
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2920481930.py", line 152, in process_case
    raise ValueError(f"No hidden arcs extracted for case {case_id}")
ValueError: No hidden arcs extracted for case ns_elephant_100


  ok | k=70 | averaged=25/25
[013/040] Processing ns_elephant_350
  failed | No hidden arcs extracted for case ns_elephant_350
[014/040] Processing ns_elephant_47


...

Traceback (most recent call last):
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2645192461.py", line 16, in <module>
    row = process_case(case_dir)
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2920481930.py", line 152, in process_case
    raise ValueError(f"No hidden arcs extracted for case {case_id}")
ValueError: No hidden arcs extracted for case s_fish_4


  ok | k=34 | averaged=25/25
[035/040] Processing s_fish_600
  ok | k=11 | averaged=25/25
[036/040] Processing s_fish_700
  ok | k=4 | averaged=5/25
[037/040] Processing s_fish_800
  ok | k=19 | averaged=25/25
[038/040] Processing s_fish_900
  ok | k=40 | averaged=25/25
[039/040] Processing s_oyster_350
  failed | No hidden arcs extracted for case s_oyster_350
[040/040] Processing s_oyster_4


Traceback (most recent call last):
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2645192461.py", line 16, in <module>
    row = process_case(case_dir)
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2920481930.py", line 152, in process_case
    raise ValueError(f"No hidden arcs extracted for case {case_id}")
ValueError: No hidden arcs extracted for case s_oyster_350


  failed | No hidden arcs extracted for case s_oyster_4


Traceback (most recent call last):
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2645192461.py", line 16, in <module>
    row = process_case(case_dir)
  File "/data/storage-occ-v2/pip-tmp/ipykernel_4018476/2920481930.py", line 152, in process_case
    raise ValueError(f"No hidden arcs extracted for case {case_id}")
ValueError: No hidden arcs extracted for case s_oyster_4


In [61]:
# %%
# -----------------------------------------------------------------------------
# Save and inspect manifest
# -----------------------------------------------------------------------------
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print("Done.")
print(f"Manifest saved to:\n{MANIFEST_CSV}")

display(manifest_df)

Done.
Manifest saved to:
/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32/mean_top1_cluster_contours_size32_manifest.csv


,case_id,source_model,occluder_size,cluster_rank,target_k,n_shapes_listed_in_cluster,n_shapes_averaged,n_shapes_failed,baseGrid_case,jsonl_path,npz_path,plot_path,members_csv_path,status,error
0,ns_cow_202,resnet50_geirhos_tl,16,1,72.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
1,ns_cow_205,resnet50_geirhos_tl,16,1,70.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
2,ns_cow_350,resnet50_geirhos_tl,16,1,56.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
3,ns_cow_510,resnet50_geirhos_tl,16,1,36.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
4,ns_cow_780,resnet50_geirhos_tl,16,1,51.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
5,ns_dog_105,resnet50_geirhos_tl,16,1,76.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
6,ns_dog_214,resnet50_geirhos_tl,16,1,9.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
7,ns_dog_350,resnet50_geirhos_tl,16,1,41.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
8,ns_dog_69,resnet50_geirhos_tl,16,1,4.0,25.0,25,0.0,256.0,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
9,ns_dog_780,resnet50_geirhos_tl,16,1,NaN,NaN,0,NaN,NaN,,,,,failed,Missing shapes_xy.npz: /home/hschatzle/data/st...


# Plotting

In [62]:
# %%
# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
RESULTS_ROOT = Path(
    "/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/"
    "bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32"
)

PLOT_DIRNAME = "mean_contour_plots"
PLOT_MANIFEST_CSV = RESULTS_ROOT / "mean_top1_cluster_contour_plot_manifest.csv"

print("RESULTS_ROOT exists:", RESULTS_ROOT.exists())
print("PLOT_MANIFEST_CSV:", PLOT_MANIFEST_CSV)

RESULTS_ROOT exists: True
PLOT_MANIFEST_CSV: /home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32/mean_top1_cluster_contour_plot_manifest.csv


In [63]:
# %%
# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def read_jsonl_rows(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def first_jsonl_row(path):
    rows = read_jsonl_rows(path)
    if len(rows) == 0:
        raise ValueError(f"No rows found in {path}")
    return rows[0]


def close_xy_if_needed(xy, tol=1e-8):
    xy = np.asarray(xy, dtype=float)
    if len(xy) == 0:
        return xy
    if np.linalg.norm(xy[0] - xy[-1]) <= tol:
        return xy
    return np.vstack([xy, xy[0]])


def style_axes_equal(ax):
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

In [64]:
# %%
# -----------------------------------------------------------------------------
# Find saved mean-contour JSONLs
# -----------------------------------------------------------------------------
jsonl_paths = sorted([
    p
    for p in RESULTS_ROOT.rglob("*_size32_top1cluster_mean_contour_k*.jsonl")
    if p.is_file()
])

print("Found mean-contour JSONLs:", len(jsonl_paths))
jsonl_paths[:5]

Found mean-contour JSONLs: 34


[PosixPath('/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32/ns_cow_202/ns_cow_202_size32_top1cluster_mean_contour_k72.jsonl'),
 PosixPath('/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32/ns_cow_205/ns_cow_205_size32_top1cluster_mean_contour_k70.jsonl'),
 PosixPath('/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32/ns_cow_350/ns_cow_350_size32_top1cluster_mean_contour_k56.jsonl'),
 PosixPath('/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_32/ns_cow_510/ns_cow_510_size32_top1cluster_mean_contour_k36.jsonl'),
 PosixPath('/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/resu

In [65]:
# %%
# -----------------------------------------------------------------------------
# One-case plot function
# -----------------------------------------------------------------------------
def plot_saved_mean_contour_jsonl(jsonl_path: Path, save_dirname=PLOT_DIRNAME):
    record = first_jsonl_row(jsonl_path)

    case_id = record["case_id"]
    target_k = int(record["target_k"])

    full_mean_xy = np.asarray(record["global_shape_filled_mean_xy"], dtype=float)
    occluder_xy = np.asarray(record["occluder_xy"], dtype=float)

    full_mean_xy = close_xy_if_needed(full_mean_xy)
    occluder_xy = close_xy_if_needed(occluder_xy)

    out_dir = jsonl_path.parent / save_dirname
    out_dir.mkdir(parents=True, exist_ok=True)

    out_png = out_dir / f"{case_id}_size16_top1cluster_mean_contour_k{target_k}_fullshape.png"

    fig, ax = plt.subplots(figsize=(8, 8))

    # Full mean contour as one continuous contour
    ax.plot(
        full_mean_xy[:, 0],
        full_mean_xy[:, 1],
        lw=2.5,
        color="black",
    )

    # Occluder overlay
    ax.plot(
        occluder_xy[:, 0],
        occluder_xy[:, 1],
        "--",
        lw=1.5,
        color="0.5",
    )

    # Clean figure: no title, no axes, no frame
    ax.set_aspect("equal")
    ax.axis("off")

    # Tight limits with a small padding
    all_xy = np.vstack([full_mean_xy, occluder_xy])
    xmin, ymin = all_xy.min(axis=0)
    xmax, ymax = all_xy.max(axis=0)

    dx = xmax - xmin
    dy = ymax - ymin
    pad_x = 0.03 * dx if dx > 0 else 1.0
    pad_y = 0.03 * dy if dy > 0 else 1.0

    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)

    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    fig.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0,
        facecolor="white",
        edgecolor="none",
    )
    plt.close(fig)

    return {
        "case_id": case_id,
        "target_k": target_k,
        "source_jsonl": str(jsonl_path),
        "output_png": str(out_png),
        "status": "ok",
        "error": "",
    }

In [66]:
# %%
# -----------------------------------------------------------------------------
# Test one example first
# -----------------------------------------------------------------------------
test_path = jsonl_paths[0]
test_result = plot_saved_mean_contour_jsonl(test_path)
pd.DataFrame([test_result])

,case_id,target_k,source_jsonl,output_png,status,error
0,ns_cow_202,72,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,


In [67]:
# %%
# -----------------------------------------------------------------------------
# Batch plot all saved mean-contour JSONLs
# -----------------------------------------------------------------------------
plot_rows = []

for i, jsonl_path in enumerate(jsonl_paths, start=1):
    print(f"[{i:03d}/{len(jsonl_paths):03d}] Plotting {jsonl_path.name}")

    try:
        row = plot_saved_mean_contour_jsonl(jsonl_path)
        plot_rows.append(row)
        print("  ok")
    except Exception as e:
        plot_rows.append({
            "case_id": "",
            "target_k": np.nan,
            "source_jsonl": str(jsonl_path),
            "output_png": "",
            "status": "failed",
            "error": str(e),
        })
        print("  failed:", e)

plot_manifest_df = pd.DataFrame(plot_rows)
plot_manifest_df.to_csv(PLOT_MANIFEST_CSV, index=False)

print("\nSaved plot manifest to:")
print(PLOT_MANIFEST_CSV)

display(plot_manifest_df)

[001/034] Plotting ns_cow_202_size32_top1cluster_mean_contour_k72.jsonl
  ok
[002/034] Plotting ns_cow_205_size32_top1cluster_mean_contour_k70.jsonl
  ok
[003/034] Plotting ns_cow_350_size32_top1cluster_mean_contour_k56.jsonl
  ok
[004/034] Plotting ns_cow_510_size32_top1cluster_mean_contour_k36.jsonl
  ok
[005/034] Plotting ns_cow_780_size32_top1cluster_mean_contour_k51.jsonl
  ok
[006/034] Plotting ns_dog_105_size32_top1cluster_mean_contour_k76.jsonl
  ok
[007/034] Plotting ns_dog_214_size32_top1cluster_mean_contour_k9.jsonl
  ok
[008/034] Plotting ns_dog_350_size32_top1cluster_mean_contour_k41.jsonl
  ok
[009/034] Plotting ns_dog_69_size32_top1cluster_mean_contour_k4.jsonl
  ok
[010/034] Plotting ns_elephant_109_size32_top1cluster_mean_contour_k70.jsonl
  ok
[011/034] Plotting ns_elephant_47_size32_top1cluster_mean_contour_k38.jsonl
  ok
[012/034] Plotting ns_elephant_780_size32_top1cluster_mean_contour_k12.jsonl
  ok
[013/034] Plotting ns_lion_350_size32_top1cluster_mean_contour_k5

,case_id,target_k,source_jsonl,output_png,status,error
0,ns_cow_202,72,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
1,ns_cow_205,70,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
2,ns_cow_350,56,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
3,ns_cow_510,36,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
4,ns_cow_780,51,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
5,ns_dog_105,76,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
6,ns_dog_214,9,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
7,ns_dog_350,41,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
8,ns_dog_69,4,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
9,ns_elephant_109,70,/home/hschatzle/data/storage-occ-v2/repos/mont...,/home/hschatzle/data/storage-occ-v2/repos/mont...,ok,
